In [1]:
!pip install albumentations
!pip install ultralytics

In [2]:
!pip install torchvision

In [3]:
!pip install Pillow albumentations matplotlib torch torchvision ultralytics tqdm wandb

In [32]:
import glob
from xml.etree import ElementTree as ET
import os
import shutil
from PIL import Image
from pathlib import Path
import pickle
import torchvision.transforms.v2


import albumentations as A
import matplotlib.pyplot as plt
import numpy as np
import torch
import torchvision
import ultralytics
from albumentations.pytorch.transforms import ToTensorV2
from matplotlib.patches import Rectangle
from torch import nn
from torchvision.models import ResNet50_Weights
from tqdm.notebook import tqdm
import logging
import time
import pandas as pd
import yaml


In [5]:
from dotenv import load_dotenv
import wandb

load_dotenv()

WANDB_API_KEY = os.getenv("WANDB_API_KEY")
WANDB_PROJECT = os.getenv("WANDB_PROJECT", "gp5")
WANDB_ENTITY = os.getenv("WANDB_ENTITY")

In [6]:
def stop_logging():
    logger = logging.getLogger()
    for handler in logger.handlers:
        handler.flush()
        handler.close()
        logger.removeHandler(handler)

def new_log_file(prefix="cnn"):
    stop_logging()
    timestamp = str(time.time()).replace('.', '_')
    log_file = f'{prefix}_{timestamp}.log'
    logging.basicConfig(
        filename=log_file,
        level=logging.INFO,
        format='%(asctime)s - %(levelname)s - %(message)s',
        force=True
    )
    logging.info("Начал логгировать новый запуск")
    return log_file

In [7]:

import sys
print(sys.executable)
print(sys.version)

/Users/alinadriagina/miniconda3/bin/python
3.12.2 | packaged by conda-forge | (main, Feb 16 2024, 20:54:21) [Clang 16.0.6 ]


In [8]:
wandb.login(key=WANDB_API_KEY)

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /Users/alinadriagina/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


True

Правильно раскидываем файлики по папочкам

In [ ]:
'''source_dir = "dataset/train"
images_dir = os.path.join(source_dir, "images")
texts_dir = os.path.join(source_dir, "labels")

os.makedirs(images_dir, exist_ok=True)
os.makedirs(texts_dir, exist_ok=True)

for filename in os.listdir(source_dir):
    filepath = os.path.join(source_dir, filename)
    ext = os.path.splitext(filename)[1].lower()

    if ext == ".jpg":
        shutil.move(filepath, os.path.join(images_dir, filename))
    elif ext == ".txt":
        shutil.move(filepath, os.path.join(texts_dir, filename))'''

In [ ]:

# for dir in ['dataset/train/images', 'dataset/train/labels', 'dataset/test/images', 'dataset/test/labels', 'dataset/val/images', 'dataset/val/labels']:
#     print(f"В папке {dir} - {len(os.listdir(dir))} файлов")

In [ ]:
'''
source_dir = "dataset/test"
images_dir = os.path.join(source_dir, "images")
texts_dir = os.path.join(source_dir, "labels")

os.makedirs(images_dir, exist_ok=True)
os.makedirs(texts_dir, exist_ok=True)

for filename in os.listdir(source_dir):
    filepath = os.path.join(source_dir, filename)
    ext = os.path.splitext(filename)[1].lower()

    if ext == ".jpg":
        shutil.move(filepath, os.path.join(images_dir, filename))
    elif ext == ".txt":
        shutil.move(filepath, os.path.join(texts_dir, filename))'''

In [81]:
def delete_classes(classes, dir_path):
    for filename in os.listdir(dir_path):
        filepath = os.path.join(dir_path, filename)

        for c in classes:
            if str(c) in filename:
                os.remove(filepath)
                break


In [83]:
delete_classes(["lemon", "chilli"], "dataset/train/images")
delete_classes(["lemon", "chilli"], "dataset/train/labels")
delete_classes(["lemon", "chilli"], "dataset/test/images")
delete_classes(["lemon", "chilli"], "dataset/test/labels")

In [84]:
new_labels = {
    0: 0,   # Bananas
    1: 1,   # Bananas (bag)
    2: 2,   # Blackberries
    3: 3,   # Raspberries
    # 4, 5 — Lemons — удалены
    6: 4,   # Grapes
    7: 5,   # Grapes (bag)
    8: 6,   # Tomatoes
    9: 7,   # Tomatoes (bag)
    10: 8,  # Apples
    11: 9,  # Apples (bag)
    # 12, 13 — Chilli — удалены
}

In [85]:
def reset_labels(dir_path):
    for file in os.listdir(dir_path):
        path = os.path.join(dir_path, file)
        new_bboxes = []
        with open(path, "r") as f:
            print(path)
            for line in f:
                parts = line.split()
                cl = int(parts[0])
                parts[0] = str(new_labels[cl])
                parts = " ".join(parts)
                new_bboxes.append(parts)

        with open(path, "w") as f:
            f.write("\n".join(new_bboxes))

In [86]:
reset_labels("dataset/train/labels")
reset_labels("dataset/test/labels")

dataset/train/labels/56_6_tomato_wob_3.txt
dataset/train/labels/13_3_banana_wb_14.txt
dataset/train/labels/128_1_grapes_wb_7.txt
dataset/train/labels/530_1_tomato_wb_51.txt
dataset/train/labels/337_4_grapes_wob_18.txt
dataset/train/labels/650_6_apple_wob_17.txt
dataset/train/labels/487_0_blackberries - 2.txt
dataset/train/labels/57_1_grapes_wob_2.txt
dataset/train/labels/546_0_banana_wb_23.txt
dataset/train/labels/7_1_apple_wob_32.txt
dataset/train/labels/288_6_tomato_wob_38.txt
dataset/train/labels/652_4_apple_wb_26.txt
dataset/train/labels/477_5_blackberries - 15.txt
dataset/train/labels/326_3_tomato_wob_20.txt
dataset/train/labels/400_5_tomato_wb_37.txt
dataset/train/labels/625_3_grapes_wb_42.txt
dataset/train/labels/342_0_grapes_wb_19.txt
dataset/train/labels/15_0_banana_wob_38.txt
dataset/train/labels/396_0_grapes_wob_23.txt
dataset/train/labels/151_3_banana_wob_28.txt
dataset/train/labels/227_1_grapes_wob_12.txt
dataset/train/labels/46_3_apple_wb_28.txt
dataset/train/labels/41_4_

In [87]:
source_dir = "dataset/train"

df_dict = {'name': [], 'class': []}
for image_path in sorted(os.listdir(os.path.join(source_dir, 'images'))):
    raw_name = os.path.splitext(image_path)[0].lower()
    if raw_name != ".ds_store":
        df_dict['name'].append(raw_name)
        with open(os.path.join(source_dir, "labels", raw_name+".txt")) as f:
            df_dict['class'].append(f.readline().split()[0])

df = pd.DataFrame(df_dict)
df 
        

,name,class
0,0_1_apple_wb_7,8
1,0_2_apple_wb_7,8
2,0_3_apple_wb_7,8
3,0_5_apple_wb_7,8
4,0_6_apple_wb_7,8
...,...,...
2689,99_1_raspberry - 17,3
2690,99_2_raspberry - 17,3
2691,99_3_raspberry - 17,3
2692,99_4_raspberry - 17,3


In [88]:
from sklearn.model_selection import train_test_split

y = df['class']
X = df.drop('class', axis=1)

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [89]:

X_val


,name
2231,585_3_tomato_wob_44
1368,402_6_tomato_wob_32
169,140_3_tomato_wob_5
584,233_2_banana_wb_49
1249,376_5_grapes_wob_27
...,...
2037,543_2_banana_wob_27
89,120_5_apple_wob_35
1962,525_1_tomato_wob_40
1119,34_6_apple_wb_16


In [90]:
train_dir = "dataset/train"
images_train_dir = os.path.join(train_dir, "images")
texts_train_dir = os.path.join(train_dir, "labels")

source_dir = "dataset/val"
if os.path.exists(source_dir):
    shutil.rmtree(source_dir)

os.makedirs(source_dir)
images_dir = os.path.join(source_dir, "images")
texts_dir = os.path.join(source_dir, "labels")

os.makedirs(images_dir, exist_ok=True)
os.makedirs(texts_dir, exist_ok=True)

for raw_name in X_val['name']:
    text_file = os.path.join(texts_train_dir, raw_name+".txt")
    img_file = os.path.join(images_train_dir, raw_name+".jpg")
    shutil.move(text_file, os.path.join(texts_dir, raw_name+".txt"))
    shutil.move(img_file, os.path.join(images_dir, raw_name+".jpg"))

In [91]:
for dir in ['dataset/train/images', 'dataset/train/labels', 'dataset/test/images', 'dataset/test/labels', 'dataset/val/images', 'dataset/val/labels']:
    print(f"В папке {dir} - {len(os.listdir(dir))} файлов")

В папке dataset/train/images - 2156 файлов
В папке dataset/train/labels - 2155 файлов
В папке dataset/test/images - 449 файлов
В папке dataset/test/labels - 449 файлов
В папке dataset/val/images - 539 файлов
В папке dataset/val/labels - 539 файлов


Напишем функцию, которая по изображению находит его описание в папке bboxes и переводит это описание в бодее удобные для работы метки.

В датасете они даны в формате "Класс Центр_прямоугольника_х, Центр_прямоугольника_y, Ширина, Высота". Мы хотим работать с библиотекой `albumentations`, там нужны метки в формате "Левый верхний_х, Левый верхний_y, Правый_нижний_х, Правый_нижний_y"

In [92]:
def get_yolo_data(image_path):
    image_path = Path(image_path)
    txt_path = str(image_path).replace("images", "labels").replace("jpg", "txt")
    img_w, img_h = Image.open(image_path).size

    bboxes = []
    with open(txt_path, "r") as f:
        for line in f:
            cls, xc, yc, w, h = map(float, line.split())
            cls = int(cls)

            if cls not in new_labels:
                continue

            xmin = int((xc - w / 2) * img_w)
            ymin = int((yc - h / 2) * img_h)
            xmax = int((xc + w / 2) * img_w)
            ymax = int((yc + h / 2) * img_h)

            bboxes.append([xmin, ymin, xmax, ymax, cls])

    return bboxes

Делаем класс, который будет хранить наш датасет

In [93]:
class PascalDataset(torch.utils.data.Dataset):
    def __init__(self, *, transform, root="dataset", mode="Train", seed=42):
        self.root = Path(root)
        self.transform = transform

        if mode == "Train":
            filenames = glob.glob(root + "/train/images/*")
        elif mode == "Test":
            filenames = glob.glob(root + "/test/images/*")
        elif mode == "Val":
            filenames = glob.glob(root + '/val/images/*')
        
        np.random.seed(seed)

    def __getitem__(self, idx):
        fname = self.filenames[idx]
        image = np.array(Image.open(fname))
        bboxes = get_yolo_data(fname)

        return self.transform(image=image, bboxes=bboxes)

    def __get_raw_item__(self, idx):
        fname = self.filenames[idx]
        return fname, get_yolo_data(fname)

    def __len__(self):
        return len(self.filenames)

Нормализуем и приведем к 512*512. Mean и std возьмем те, с которыми обучался ResNet, так как в будущем мы будем его использовать


In [94]:
mean = (0.485, 0.456, 0.406)
std = (0.229, 0.224, 0.225)

train_transform = A.Compose(
    [
        A.Resize(512, 512),
        A.Normalize(mean=mean, std=std),
        ToTensorV2(),
    ],
    bbox_params=dict(format="pascal_voc", min_visibility=0.3),
)

test_transform = A.Compose(
    [
        A.Resize(512, 512),
        A.Normalize(mean=mean, std=std),
        ToTensorV2(),
    ],
    bbox_params=dict(format="pascal_voc", min_visibility=0.5),
)

Создаем датасеты

In [95]:
train_ds = PascalDataset(root="dataset/", transform=train_transform, mode="Train")
val_ds = PascalDataset(root="dataset/", transform=test_transform, mode="Val")
test_ds = PascalDataset(root="dataset/", transform=test_transform, mode="Test")

Словарь, который переводит метку класса в название класса для визуализации. Одному классу принадлежит 2 метки, так как у фруктов есть две разновидности фотографий "в пакете" и "без пакета"

Функуция визуализации картинки с квадратиками

In [96]:
class_labels = {
    0: "Bananas",
    1: "Bananas",
    2: "Blackberries",
    3: "Raspberries",
    4: "Grapes",
    5: "Grapes",
    6: "Tomatoes",
    7: "Tomatoes",
    8: "Apples",
    9: "Apples",
}

In [97]:
def visualize(images, bboxes):
    mean = (0.485, 0.456, 0.406)
    std = (0.229, 0.224, 0.225)

    fig, axes = plt.subplots(2, len(images) // 2 + len(images) % 2, figsize=(10, 8))

    for i, ax in enumerate(axes.reshape(-1)):

        ax.axis(False)

        if i >= len(images):
            break
        
        # денормализуем - возвращаем изображению изначальные цвета
        image = images[i]
        image = torch.permute(image, (1, 2, 0)).numpy()
        image = image * std + mean
        image = np.clip(image, 0, 1)

        ax.imshow(image)

        for bbox in bboxes[i]:
          # при показе готового фото из датасета с готовой разметкой, там нет уверенности в классе
          if len(bbox) == 5:
            xmin, ymin, xmax, ymax, cl = bbox
          # при показе прогноза, есть уверенность в классе
          if len(bbox) == 6:
            xmin, ymin, xmax, ymax, conf, cl = bbox
          rectangle = plt.Rectangle((xmin, ymin), xmax-xmin, ymax-ymin, fill=False, color="m")
          ax.add_patch(rectangle)
          ax.text(xmin, ymin-10, class_labels[cl], color="m", fontweight="bold")

    fig.tight_layout()
    plt.show()

Напишем функцию collate_fn, которая выполняет преобразование bounding boxes в формат карты признаков для детекции объектов.
Она делит изображение на 16 квадратов по вертикали и по горизонтали (в нашем датасете сторона квадрата по 32 пикселя).
Для каждого квадрата указываем 6 характеристик:
1. Относительный сдвиг центра bounding box относительно размера квадрата по Х (центр находится на 40% длины квадрата)
2. Относительный сдвиг центра bounding box относительно размера квадрата по Y (центр находится на 60% высоты квадрата)
3. Нормализованная ширина bounding box (он занимает 50% квадрата по ширине)
4. Нормализованная высота bounding box (он занимает 45% квадрата по высоте)
5. Confidence сетки - насколько мы уверены, что в этой клетке есть bbox
6. Класс детекции

![image](https://i.imgur.com/13YVxAd.jpeg)



In [98]:
def collate_fn(batch, pieces=(16, 16)):

    imgs = []
    batch_boxes = []

    for b in batch:
        imgs.append(b["image"])
        batch_boxes.append(b["bboxes"])

    imgs = torch.stack(imgs)
    b, c, h, w = imgs.shape

    if isinstance(pieces, int):
        pieces_h, pieces_w = pieces, pieces
    else:
        pieces_h, pieces_w = pieces

    ds_h = h // pieces_h
    ds_w = w // pieces_w

    target = imgs.new_zeros(b, 6, pieces_h, pieces_w)

    for i in range(len(batch_boxes)):
        boxes = imgs.new_tensor(batch_boxes[i])

        xmin, ymin, xmax, ymax, classes = boxes.T

        #считаем относительные w и h (3 и 4 каналы)
        w_box = (xmax-xmin)/w
        h_box = (ymax-ymin)/h

        #координаты центра в абсолютных значениях
        cx = (xmax + xmin) / 2
        cy = (ymax + ymin) / 2

        # считаем в какой квадрат попал центр bbox
        cx_idx = (cx // ds_w).long()
        cy_idx = (cy // ds_h).long()

        # считаем относительные сдвиги (1 и 2 каналы)
        cx_box = (cx - ds_w * cx_idx) / ds_w
        cy_box = (cy - ds_h * cy_idx) / ds_h

        # собираем каналы для одной клетки, conf пока равно 1
        target[i, :, cy_idx, cx_idx] = torch.stack(
            [cx_box, cy_box, w_box, h_box, torch.ones_like(cx_box), classes]
        )

    return {"image": imgs, "target": target}


## ПРОВЕСТИ ПЕРВЫЙ ЭКСПЕРИМЕНТ С САМОПИСНОЙ МОДЕЛЬЮ!!!!

Для начала напишем небольшую сверточную сеть для решения этой задачи

In [ ]:
# САМОПИСНЫЙ КЛАСС МОДЕЛЬКИ. 
class SimpleDetector(nn.Module):
    def __init__(self, C):
        super().__init__()
        
        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            
            nn.Conv2d(256, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
        )
        
        self.output_layer = nn.Sequential(
            nn.Conv2d(512, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 5 + C, kernel_size=1),
            nn.Sigmoid() 
        )
        
    def forward(self, x):
        x = self.conv_layers(x)
        x = self.output_layer(x)
        return x

Напишем функцию потерь аналогичную той, что была в начальных версиях YOLO 

Она состоит из 4-х основных компонентов:

1. localization loss - MSE по координатам бокса там, в боксе, где есть детектируемый объект (нашла ли модель объект в принципе)
2. box_loss - MSE от корней ширины и высоты bbox там, где есть детектируемый объект (как точно модель выделила объект рамочкой)
3. classification_loss - если детектируемый объект есть, то его кросс-энтропия по его классу (насколько правильно модель определила класс объекта)
4. confidence_loss - бинарная кросс-энтропия факта наличия объекта в квадрате. Учитываем верное угадывание, что объекта в квадрате нет, но с меньшим весом, чтобы модель не шла в сторону сильных ложноотрицательных результатов

In [ ]:

def special_loss(pred, target, C=10):

    mask_obj = target[4] == 1

    loss_cx_box = torch.nn.functional.mse_loss(torch.masked_select(pred[0], mask_obj), torch.masked_select(target[0], mask_obj), reduction="sum")
    loss_cy_box = torch.nn.functional.mse_loss(torch.masked_select(pred[1], mask_obj), torch.masked_select(target[1], mask_obj), reduction="sum")
    localization_loss = loss_cx_box+loss_cy_box

    loss_w = torch.nn.functional.mse_loss(torch.sqrt(torch.masked_select(pred[2], mask_obj)), torch.sqrt(torch.masked_select(target[2], mask_obj)), reduction="sum")
    loss_h = torch.nn.functional.mse_loss(torch.sqrt(torch.masked_select(pred[3], mask_obj)), torch.sqrt(torch.masked_select(target[3], mask_obj)), reduction="sum")
    box_loss = loss_w+loss_h

    arr = []
    for i in range(5, 5 + C):
        arr.append(torch.masked_select(pred[i], mask_obj).unsqueeze(1))
    cls_pred = torch.cat(arr, dim=1) 
    cls_target = torch.masked_select(target[5], mask_obj).long()
    classification_loss = torch.nn.functional.cross_entropy(cls_pred, cls_target, reduction="sum")

    bce = nn.BCELoss(reduction="sum")
    yes_objects = bce(torch.masked_select(pred[4], mask_obj), torch.masked_select(target[4], mask_obj))
    no_objects = bce(torch.masked_select(pred[4], ~mask_obj), torch.masked_select(target[4], ~mask_obj))
    confidence_loss = yes_objects + 0.1*no_objects

    return localization_loss + box_loss + classification_loss + confidence_loss

In [ ]:
loader = torch.utils.data.DataLoader(train_ds, 10, collate_fn=collate_fn, shuffle=True)

In [ ]:
val_loader = torch.utils.data.DataLoader(val_ds, 10, collate_fn=collate_fn, shuffle=True)

In [ ]:
try:
    if torch.cuda.is_available():
        device = torch.device("cuda")
    elif torch.mps.is_available():
        device =torch.device("mps")
    else:
        device = torch.device("cpu")
except AttributeError:
    device = torch.device("cpu")

device

In [ ]:
!pip install torchmetrics

In [ ]:
def target_to_metric_format(targets, img_size=(512, 512)):
    pass

In [ ]:
def preds_to_metric_format(pred, threshold=0.5):
    pass


In [ ]:
from torchmetrics.detection.mean_ap import MeanAveragePrecision




def evaluate_model(model, loader, threshold=0.5, name="val"):
    model.eval()

    metric = MeanAveragePrecision(iou_type="bbox")
    losses = []

    with torch.no_grad():
        for batch in tqdm(loader, desc=f"Evaluate {name}", leave=False):
            images = batch["image"].to(device)
            targets = batch["target"].to(device)

            pred = model(images)

            preds_metric = preds_to_metric_format(pred, threshold=threshold)
            targets_metric = target_to_metric_format(targets)

            metric.update(preds_metric, targets_metric)

    metrics = metric.compute()

    return {
        "dataset": name,
        "map": metrics["map"].item(),
        "map_50": metrics["map_50"].item(),
        "map_75": metrics["map_75"].item(),
    }

In [ ]:
import copy

def train_model(model, train_loader, val_loader, loss_fn, optimizer, 
    epochs=30, threshold=0.5,model_name="model"):
    best_metric = -10**9
    best_epoch = 0
    best_state = None

    history = []
    logging.info(f"Начали обучение {model_name}")

    for e in tqdm(range(1, epochs + 1), desc="Epoch:"):
        model.train()
        epoch_loss = 0
        v = 0
        for batch in tqdm(train_loader, desc=f"Epoch {e}", leave=False):
            images = batch["image"].to(device)
            targets = batch["target"].to(device)
            v += 1
            if v == 5:
                break
            optimizer.zero_grad()
            pred = model(images)
            
            batch_loss = 0
            for el in range(pred.shape[0]):
                batch_loss += loss_fn(pred[el], targets[el], C=C)
            
            batch_loss /= pred.shape[0]
            batch_loss.backward()
            epoch_loss += batch_loss.item()
            optimizer.step()

        epoch_loss /= len(train_loader)
        print(threshold)
        valid_metrics = evaluate_model(model=model, loader=val_loader, threshold=threshold,
            name=f"{model_name} | valid epoch {e}")
        

        if valid_metrics["map_50"] > best_metric:
            best_metric = valid_metrics["map_50"]
            best_epoch = e
            best_state = copy.deepcopy(model.state_dict())

        print(f"Epoch {e} done; Train loss {epoch_loss:.3f}")
        print(valid_metrics)
        logging.info(
            f"Эпоха {e} | "
            f"Train Loss: {epoch_loss:.4f} | "
            f"Val Map: {valid_metrics['map']:.4f} | "
            f"Val Map50: {valid_metrics['map_50']:.4f}"
        )
        f = {"train/loss": epoch_loss, **valid_metrics}
        wandb.log(f)
    
    model.load_state_dict(best_state)
    
    return model

In [ ]:
def save_results(model, name, train_loader, test_loader, log_file, run):
    train_metrics = evaluate_model(model, train_loader, threshold=0.5, name="Train")
    test_metrics = evaluate_model(model, test_loader, threshold=0.5, name="Test")
    pickle.dump(model.state_dict(), open(f"models/{name}.pkl", 'wb'))
    logging.info("Сохранили веса модели в папку models")

    metric_keys = list(train_metrics.keys())

    table = wandb.Table(columns=metric_keys)
    table.add_data(*[train_metrics[i] for i in metric_keys])
    table.add_data(*[test_metrics[i] for i in metric_keys])
    wandb.log({'results': table})
    artifact = wandb.Artifact(name=name, type="model", description=f"Тест логгирования модели: {name}")

    artifact.add_file(f"models/{name}.pkl")
    artifact.add_file(log_file)
    run.log_artifact(artifact) 

In [ ]:
LR = 1e-3
EPOCHS_1 = 10
BATCH_SIZE = 10


log_file = new_log_file()


model_test_1 = SimpleDetector(C).to(device)
opt_1 = torch.optim.Adam(model_test_1.parameters(), lr=LR)
run_exp1 = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY, name="simple_detector_exp1", config=config_exp1)
loss = special_loss
loader = torch.utils.data.DataLoader(train_ds, BATCH_SIZE, collate_fn=collate_fn, shuffle=True)


config_exp1 = {
    "model": "SimpleDetector",
    "epochs": EPOCHS_1,
    "batch_size": BATCH_SIZE,
    "learning_rate": LR,
    "seed": 21,
    "loss": loss.__class__.__name__,
    "architecture": str(model_test_1),
    "optimitzer": opt_1.__class__.__name__,
    "task": "detection",
}

m = train_model(model=model_test_1, train_loader=loader, val_loader=val_loader, loss_fn=special_loss, optimizer=opt_1, epochs=1, threshold=0.5)
test_loader = torch.utils.data.DataLoader(test_ds, 10, collate_fn=collate_fn, shuffle=True)
save_results(m, "test", loader, test_loader, log_file, run_exp1)
run_exp1.finish()
#train_model()

Напишем функцию, которая переводит предсказания модели в удобные для рисования числа. Сразу на этом этапе учтем ситуацию, когда модель предсказывает один и тот же бокс в соседних квадратах. Будем использовать IoU, чтобы не отображать менее уверенные из пересекающихся боксов.

In [27]:
# считаем iou для боксов
def iou(bbox1, bbox2):
  xmin1, ymin1, xmax1, ymax1, conf1, cl1 = bbox1
  xmin2, ymin2, xmax2, ymax2, conf2, cl2 = bbox2
  intersection = torch.clamp(torch.min(xmax1, xmax2) - torch.max(xmin1, xmin2), min=0) * torch.clamp(torch.min(ymax1, ymax2) - torch.max(ymin1, ymin2), min=0)
  intersection = torch.clamp(intersection, min=0)
  union = (xmax1-xmin1)*(ymax1-ymin1) + (xmax2-xmin2)*(ymax2-ymin2) - intersection
  return intersection/union

# выкидываем менее уверенные боксы
def NMS(bboxes, threshold):
    leave = []
    classes = bboxes[:, 5].unique()

    for cls in classes:
        cls_bboxes = bboxes[bboxes[:, 5] == cls]
        cls_bboxes = cls_bboxes[torch.argsort(cls_bboxes[:, 4], descending=True)]

        while len(cls_bboxes) > 0:
            leave.append(cls_bboxes[0])
            left_check = []
            for box in cls_bboxes[1:]:
                if iou(cls_bboxes[0], box) <= threshold:
                    left_check.append(box)
                    
            if len(left_check) == 0:
                break
            cls_bboxes = torch.stack(left_check)

    if len(leave) == 0:
        return torch.zeros((0,6), dtype=torch.float32)

    return torch.stack(leave)


In [28]:
def decode_prediction_nms(pred, img_size=(512, 512), threshold=0.7):
    b, c, h, w = pred.shape
    img_w, img_h = img_size
    ds_h = img_h // h 
    ds_w = img_w // w
    
    cx_idx = torch.arange(w, dtype=torch.float32).reshape(1, 1, 1, w)
    cy_idx = torch.arange(h, dtype=torch.float32).reshape(1, 1, h, 1)
    result = []

    for i in range(len(pred)):
      image = pred[i]
      cx_box, cy_box, w_box, h_box, conf = image[:5]
      cl = torch.argmax(image[5:], dim=0)

      cx = cx_box*ds_w + ds_w*cx_idx
      cy = cy_box*ds_h + ds_h*cy_idx

      width = w_box * img_w
      height = h_box * img_h

      xmin = (cx - width / 2).squeeze()
      ymin = (cy - height / 2).squeeze()
      xmax = (cx + width / 2).squeeze()
      ymax = (cy + height / 2).squeeze()
      cl = cl.squeeze()

      res = torch.stack([xmin, ymin, xmax, ymax, conf, cl], dim=2)[conf.squeeze() > threshold]
      boxes_nms = NMS(res, threshold=0.6)

      result.append(boxes_nms.tolist())


    return result

Посмотрим на один батч

In [ ]:
test_loader = torch.utils.data.DataLoader(test_ds, 10, collate_fn=collate_fn)
i = iter(test_loader)
batch = next(i)

In [ ]:
model_test_1.eval()
pred = model_test_1(batch["image"].to(device)).cpu()


In [ ]:
pred_decoded = decode_prediction_nms(pred, threshold=0.5)
visualize(batch["image"], pred_decoded)

## Эксперимент 2

Проведем эксперимент с другой моделью, в этот раз возьмем слои от ResNet50, обучим исходные слои с меньшим шагом, а новые - с большим

In [ ]:
class Detector(nn.Module):
    def __init__(self, C):
        super().__init__()
        model = torchvision.models.resnet50(weights=ResNet50_Weights.DEFAULT)
        self.rn = nn.Sequential(*list(model.children())[:-2])

        self.vgg = nn.Sequential(
           nn.Conv2d(in_channels=2048, out_channels=512, kernel_size=3, padding=1),
           nn.BatchNorm2d(num_features=512),
           nn.ReLU(),

           nn.Conv2d(in_channels=512, out_channels=128, kernel_size=3, padding=1),
           nn.BatchNorm2d(num_features=128),
           nn.ReLU(),

           nn.Conv2d(in_channels=128, out_channels=32, kernel_size=3, padding=1),
           nn.BatchNorm2d(num_features=32),
           nn.ReLU(),

           nn.Conv2d(in_channels=32, out_channels=5+C, kernel_size=3, padding=1),
           nn.BatchNorm2d(num_features=5+C),
           nn.Sigmoid(),
           
        )

    def forward(self, img):
        x = self.rn(img)
        x = self.vgg(x)
        return x

In [ ]:
loader = torch.utils.data.DataLoader(train_ds, 10, collate_fn=collate_fn, shuffle=True)

In [ ]:
try:
    if torch.cuda.is_available():
        device = torch.device("cuda")
    elif torch.mps.is_available():
        device =torch.device("mps")
    else:
        device = torch.device("cpu")
except AttributeError:
    device = torch.device("cpu")

device

Обучаем модельку

In [ ]:
LR_BACKBONE=1e-5
LR_HEAD=1e-3
EPOCHS = 10
BATCH_SIZE = 1



In [ ]:
torch.manual_seed(21)
model_test_2 = Detector(C).to(device)

log_file = new_log_file()

opt = torch.optim.Adam([
    {'params': model_test_2.rn.parameters(), 'lr': LR_BACKBONE},   
    {'params': model_test_2.vgg.parameters(), 'lr': LR_HEAD},
])

loss = special_loss
config={
        "model": "Detector",
        "backbone": "ResNet50",
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "lr_backbone": LR_BACKBONE,
        "lr_head": LR_HEAD,
        "seed": 21,
        "loss": loss.__class__.__name__,
        "architecture": str(model_test_2),
        "optimizer": opt.__class__.__name__,
        "task": "detection",
}

run = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY, name="cnn_detector_model_test_2", config=config)
m = train_model(model_test_2, loader, val_loader, loss, opt, EPOCHS, model_name="ResNet")
test_loader = torch.utils.data.DataLoader(test_ds, 10, collate_fn=collate_fn, shuffle=True)
save_results(m, "test", loader, test_loader, log_file, run)
run.finish()

In [ ]:
wandb.finish()

## Эксперимент 3

Попробуем использовать уже проверенную архитектуру - YOLO из библиотеки ultralytics.

In [45]:
yolo_data_dir = 'dataset'

In [102]:
yolo_config = {
    'path': os.path.abspath(yolo_data_dir),
    'train': 'train/images',
    'val': 'val/images',
    'nc': C, 
    'names': list(class_labels.values())
}

with open(os.path.join(yolo_data_dir, 'data.yaml'), 'w', encoding='utf-8') as f:
    yaml.dump(yolo_config, f, allow_unicode=True)

print(yaml.dump(yolo_config, allow_unicode=True))

names:
- Bananas
- Bananas
- Blackberries
- Raspberries
- Grapes
- Grapes
- Tomatoes
- Tomatoes
- Apples
- Apples
nc: 10
path: /Users/alinadriagina/Desktop/gp5 git/gp-5/dataset
train: train/images
val: val/images



In [103]:
device = "mps"

In [104]:
from ultralytics import YOLO

EPOCHS = 1

model_yolo = YOLO('yolov8n.pt')

results = model_yolo.train(
    data=os.path.join(yolo_data_dir, 'data.yaml'),
    epochs=EPOCHS,
    imgsz=512,
    batch=16,
    device='mps',  
    workers=0,
    patience=10,
    project='yolo_experiment',
    name='yolov8_fruits',
    exist_ok=True,
    verbose=True,
    seed=21
)

print(f"Лучшая модель: {results.save_dir}/weights/best.pt")

Ultralytics 8.4.60 🚀 Python-3.12.2 torch-2.12.0 MPS (Apple M3)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=dataset/data.yaml, degrees=0.0, deterministic=True, device=mps, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=1, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=512, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8_fruits, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=10, perspective=0.0, plots=True

KeyboardInterrupt: 

In [37]:
from ultralytics import YOLO

EPOCHS = 10

run_yolo = wandb.init(
    project=WANDB_PROJECT, 
    entity=WANDB_ENTITY, 
    name="yolov8_experiment",
    config={
        "model": "yolov8n",
        "epochs": EPOCHS,
        "imgsz": 512,
        "batch": 16,
        "data": "dataset/data.yaml"
    }
)

model_yolo = YOLO('yolov8n.pt')

results = model_yolo.train(
    data=os.path.join(yolo_data_dir, 'data.yaml'),
    epochs=EPOCHS,
    imgsz=512,
    batch=16,
    device='cpu',  
    workers=0,
    patience=10,
    project='yolo_experiment',
    name='yolov8_fruits',
    exist_ok=True,
    verbose=True,
    seed=21
)

print(f"Лучшая модель: {results.save_dir}/weights/best.pt")

MailboxClosedError: 

In [106]:
best_model = YOLO(f'{results.save_dir}/weights/best.pt')
metrics = best_model.val(data=os.path.join(yolo_data_dir, 'data.yaml'), split='val')

print(f"  mAP@0.5: {metrics.box.map50:.4f}")
print(f"  mAP@0.5:0.95: {metrics.box.map:.4f}")
print(f"  Precision: {metrics.box.mp:.4f}")
print(f"  Recall: {metrics.box.mr:.4f}")

Ultralytics 8.4.60 🚀 Python-3.12.2 torch-2.12.0 CPU (Apple M3)
Model summary (fused): 73 layers, 3,007,598 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1644.0±464.5 MB/s, size: 1381.2 KB)
val: Scanning /Users/alinadriagina/Desktop/gp5 git/gp-5/dataset/val/labels.cache... 539 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 539/539 251.2Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 34/34 6.5s/it 3:395.8ss
                   all        539        647      0.723      0.815      0.861       0.66
               Bananas         58         58      0.425          1      0.988      0.802
               Bananas         63         75       0.82          1      0.971      0.835
          Blackberries         23         23      0.911      0.609      0.727      0.489
           Raspberries         30         30      0.634      0.933      0.931      0.594
                Grapes        

In [ ]:
best_model = YOLO(f'{results.save_dir}/weights/best.pt')
metrics = best_model.val(data=os.path.join(yolo_data_dir, 'data.yaml'), split='val')

wandb.log({
    "yolo/mAP50": metrics.box.map50,
    "yolo/mAP50-95": metrics.box.map,
    "yolo/precision": metrics.box.mp,
    "yolo/recall": metrics.box.mr
})

print(f"  mAP@0.5: {metrics.box.map50:.4f}")
print(f"  mAP@0.5:0.95: {metrics.box.map:.4f}")
print(f"  Precision: {metrics.box.mp:.4f}")
print(f"  Recall: {metrics.box.mr:.4f}")

wandb.finish()

In [ ]:
model = YOLO("yolov11n.pt")
results = model.val(
        data="yolo_dataset/data.yaml",
        imgsz=512,
        batch=16,
        device='cpu'
)
 

print(f"  mAP@0.5: {results.box.map50:.4f}")
print(f"  mAP@0.5:0.95: {results.box.map:.4f}")
    
wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY, name="exp3_test", reinit=True)
wandb.log({
        "exp3_test/mAP50": results.box.map50,
        "exp3_test/mAP50_95": results.box.map,
})
wandb.finish()

In [ ]:
model = YOLO('yolo11n.pt')  

results = model.train(
    data='dataset/dataset.yaml',
    epochs=10,
    imgsz=640,
    batch=16,
    device='cpu',
    workers=0,
    verbose=True
)



In [ ]:
metrics = model.val(data='dataset/dataset.yaml')
print(f"mAP50: {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")

model.export(format='onnx') 

In [ ]:
def predict_and_visualize_yolo(model, image_path):
    results = model(image_path)
    result = results[0]
    img = result.orig_img
    boxes = result.boxes
    
    fig, ax = plt.subplots(1, figsize=(10, 10))
    ax.imshow(img)
    ax.axis('off')
    
    if boxes is not None:
        for box in boxes:
            x1, y1, x2, y2 = box.xyxy[0].tolist()
            conf = box.conf[0].item()
            cls = int(box.cls[0].item())
            rect = plt.Rectangle((x1, y1), x2-x1, y2-y1, 
                                 fill=False, edgecolor='red', linewidth=2)
            ax.add_patch(rect)
            
            label = f"{class_labels[cls]} ({conf:.2f})"
            ax.text(x1, y1-5, label, color='red', fontsize=10, 
                   fontweight='bold', bbox=dict(facecolor='white', alpha=0.8))
    plt.show()

test_images = list(Path('dataset/test/images').glob('*.jpg'))[:3]
for img_path in test_images:
    predict_and_visualize_yolo(model, str(img_path))